# Task 1 — Data Exploration and 10% Sampling

This notebook explores the aligned English–Spanish Europarl corpus and creates a reproducible 10% sample for the later tasks.

Main goals:

1. verify that both language files are aligned,
2. calculate corpus and sentence-length statistics,
3. visualize important relationships between English and Spanish,
4. select exactly 10% of the aligned sentence pairs with a fixed random seed.


In [ ]:
from pathlib import Path
from itertools import zip_longest
from array import array
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

RANDOM_SEED = 43
SAMPLE_FRACTION = 0.10


def find_project_root(start: Path) -> Path:
    """Find the repository root by searching for the data directory."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Start Jupyter inside the repository."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
EN_PATH = DATA_DIR / "europarl-v7.es-en.en"
ES_PATH = DATA_DIR / "europarl-v7.es-en.es"

ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "task01"
FIGURE_DIR = ARTIFACT_DIR / "figures"
SAMPLE_DIR = DATA_DIR / "sampled"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

assert EN_PATH.exists(), f"Missing file: {EN_PATH}"
assert ES_PATH.exists(), f"Missing file: {ES_PATH}"

print(f"Project root: {PROJECT_ROOT}")
print(f"English file: {EN_PATH}")
print(f"Spanish file: {ES_PATH}")

## 1. Read the aligned corpus and collect statistics

The files are processed line by line so that the complete corpus does not have to be loaded into memory as text. Each English line must correspond to the Spanish line with the same index.


In [ ]:
en_token_lengths = array("I")
es_token_lengths = array("I")
en_char_lengths = array("I")
es_char_lengths = array("I")

empty_en = 0
empty_es = 0
empty_pair = 0
xml_en = 0
xml_es = 0
total_pairs = 0

with EN_PATH.open("r", encoding="utf-8", errors="replace") as en_file, \
     ES_PATH.open("r", encoding="utf-8", errors="replace") as es_file:

    aligned_lines = zip_longest(en_file, es_file, fillvalue=None)

    for index, (en_line, es_line) in enumerate(
        tqdm(aligned_lines, desc="Reading aligned sentence pairs")
    ):
        if en_line is None or es_line is None:
            raise ValueError(
                f"The two files have different numbers of lines. "
                f"Alignment failed near line {index + 1}."
            )

        en_text = en_line.strip()
        es_text = es_line.strip()

        en_is_empty = not en_text
        es_is_empty = not es_text

        empty_en += int(en_is_empty)
        empty_es += int(es_is_empty)
        empty_pair += int(en_is_empty or es_is_empty)
        xml_en += int(en_text.startswith("<"))
        xml_es += int(es_text.startswith("<"))

        en_token_lengths.append(len(en_text.split()))
        es_token_lengths.append(len(es_text.split()))
        en_char_lengths.append(len(en_text))
        es_char_lengths.append(len(es_text))

        total_pairs += 1

print(f"Aligned sentence pairs: {total_pairs:,}")
print(f"Pairs with at least one empty side: {empty_pair:,}")
print(f"English lines starting with '<': {xml_en:,}")
print(f"Spanish lines starting with '<': {xml_es:,}")

In [ ]:
en_tokens = np.asarray(en_token_lengths, dtype=np.int32)
es_tokens = np.asarray(es_token_lengths, dtype=np.int32)
en_chars = np.asarray(en_char_lengths, dtype=np.int32)
es_chars = np.asarray(es_char_lengths, dtype=np.int32)

token_difference = es_tokens - en_tokens
token_ratio = np.divide(
    es_tokens,
    en_tokens,
    out=np.full(es_tokens.shape, np.nan, dtype=np.float64),
    where=en_tokens != 0,
)


def describe(values: np.ndarray) -> dict:
    return {
        "count": int(values.size),
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "min": int(np.min(values)),
        "25%": float(np.percentile(values, 25)),
        "median": float(np.median(values)),
        "75%": float(np.percentile(values, 75)),
        "95%": float(np.percentile(values, 95)),
        "99%": float(np.percentile(values, 99)),
        "max": int(np.max(values)),
    }


length_summary = pd.DataFrame(
    {
        "English tokens": describe(en_tokens),
        "Spanish tokens": describe(es_tokens),
        "English characters": describe(en_chars),
        "Spanish characters": describe(es_chars),
    }
).T

length_summary

In [ ]:
valid_ratio = token_ratio[~np.isnan(token_ratio)]

corpus_summary = {
    "sentence_pairs": total_pairs,
    "empty_english_lines": empty_en,
    "empty_spanish_lines": empty_es,
    "pairs_with_empty_side": empty_pair,
    "english_xml_lines": xml_en,
    "spanish_xml_lines": xml_es,
    "token_length_correlation": float(np.corrcoef(en_tokens, es_tokens)[0, 1]),
    "mean_spanish_minus_english_tokens": float(np.mean(token_difference)),
    "median_spanish_to_english_token_ratio": float(np.median(valid_ratio)),
}

pd.Series(corpus_summary, name="value")

## 2. Visualizations

The plots are clipped at high percentiles where appropriate. This prevents a small number of extremely long sentences from hiding the main distribution.


In [ ]:
upper = int(np.percentile(np.concatenate([en_tokens, es_tokens]), 99))

plt.figure(figsize=(9, 5))
plt.hist(en_tokens[en_tokens <= upper], bins=60, alpha=0.6, label="English")
plt.hist(es_tokens[es_tokens <= upper], bins=60, alpha=0.6, label="Spanish")
plt.xlabel("Whitespace-separated tokens per sentence")
plt.ylabel("Number of sentences")
plt.title("Sentence-length distributions up to the 99th percentile")
plt.legend()
plt.tight_layout()

path = FIGURE_DIR / "token_length_distribution.png"
plt.savefig(path, dpi=200)
plt.show()

print(path)

In [ ]:
x_limit = int(np.percentile(en_tokens, 99))
y_limit = int(np.percentile(es_tokens, 99))
mask = (en_tokens <= x_limit) & (es_tokens <= y_limit)

plt.figure(figsize=(7, 6))
plt.hexbin(
    en_tokens[mask],
    es_tokens[mask],
    gridsize=60,
    mincnt=1,
    bins="log",
)
plt.xlabel("English tokens")
plt.ylabel("Spanish tokens")
plt.title("Relationship between aligned sentence lengths")
plt.colorbar(label="log10(sentence-pair count)")
plt.tight_layout()

path = FIGURE_DIR / "aligned_length_relationship.png"
plt.savefig(path, dpi=200)
plt.show()

print(path)

In [ ]:
lower_diff, upper_diff = np.percentile(token_difference, [1, 99])
difference_mask = (
    (token_difference >= lower_diff)
    & (token_difference <= upper_diff)
)

plt.figure(figsize=(9, 5))
plt.hist(token_difference[difference_mask], bins=60)
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("Spanish token count minus English token count")
plt.ylabel("Number of sentence pairs")
plt.title("Difference in length between aligned sentences")
plt.tight_layout()

path = FIGURE_DIR / "token_length_difference.png"
plt.savefig(path, dpi=200)
plt.show()

print(path)

In [ ]:
plot_data = [
    en_tokens[en_tokens <= np.percentile(en_tokens, 99)],
    es_tokens[es_tokens <= np.percentile(es_tokens, 99)],
]

plt.figure(figsize=(7, 5))
plt.boxplot(plot_data, tick_labels=["English", "Spanish"], showfliers=False)
plt.ylabel("Tokens per sentence")
plt.title("English and Spanish sentence lengths")
plt.tight_layout()

path = FIGURE_DIR / "token_length_boxplot.png"
plt.savefig(path, dpi=200)
plt.show()

print(path)

## 3. Create an exact, reproducible 10% sample

The same sentence indices are used for both language files, so the parallel alignment is preserved. The selected indices are stored as well, which makes the sampling step reproducible.


In [ ]:
sample_size = int(round(total_pairs * SAMPLE_FRACTION))
rng = np.random.default_rng(RANDOM_SEED)

sample_indices = np.sort(
    rng.choice(total_pairs, size=sample_size, replace=False)
)

sample_en_path = SAMPLE_DIR / "europarl_10pct.en"
sample_es_path = SAMPLE_DIR / "europarl_10pct.es"
sample_indices_path = ARTIFACT_DIR / "sample_indices.npy"

np.save(sample_indices_path, sample_indices)

selected = 0
next_position = 0

with EN_PATH.open("r", encoding="utf-8", errors="replace") as en_file, \
     ES_PATH.open("r", encoding="utf-8", errors="replace") as es_file, \
     sample_en_path.open("w", encoding="utf-8") as sample_en_file, \
     sample_es_path.open("w", encoding="utf-8") as sample_es_file:

    for index, (en_line, es_line) in enumerate(
        tqdm(zip(en_file, es_file), total=total_pairs, desc="Writing 10% sample")
    ):
        if next_position >= sample_size:
            break

        if index == sample_indices[next_position]:
            sample_en_file.write(en_line)
            sample_es_file.write(es_line)
            selected += 1
            next_position += 1

assert selected == sample_size

sampling_metadata = {
    "random_seed": RANDOM_SEED,
    "sample_fraction": SAMPLE_FRACTION,
    "corpus_sentence_pairs": total_pairs,
    "sample_sentence_pairs": selected,
    "english_sample_file": str(sample_en_path.relative_to(PROJECT_ROOT)),
    "spanish_sample_file": str(sample_es_path.relative_to(PROJECT_ROOT)),
}

metadata_path = ARTIFACT_DIR / "sampling_metadata.json"
metadata_path.write_text(
    json.dumps(sampling_metadata, indent=2),
    encoding="utf-8",
)

print(json.dumps(sampling_metadata, indent=2))

## 4. Save numerical results

The resulting CSV and JSON files can be used when writing the report.


In [ ]:
length_summary_path = ARTIFACT_DIR / "length_summary.csv"
corpus_summary_path = ARTIFACT_DIR / "corpus_summary.json"

length_summary.to_csv(length_summary_path)
corpus_summary_path.write_text(
    json.dumps(corpus_summary, indent=2),
    encoding="utf-8",
)

print(f"Saved: {length_summary_path}")
print(f"Saved: {corpus_summary_path}")
print(f"Saved figures in: {FIGURE_DIR}")
print(f"Saved sampled corpus in: {SAMPLE_DIR}")

## 5. Findings for the report

After executing the notebook, summarize the most important observations here. Useful points may include:

- the total number of aligned sentence pairs,
- whether empty or XML-like lines occur,
- typical and extreme sentence lengths,
- whether Spanish sentences are generally longer or shorter than their English counterparts,
- how strongly the lengths of aligned sentences correlate,
- whether there are unusual outliers that should be handled during preprocessing.
